# 实验五：Jetson Nano 实时四色目标识别（双画面版）

参考《2.1.在OpenCV 中调用摄像头.ipynb》的 `widgets.Image → JPEG → value` 显示方式。**最终只显示两幅实时图像：原始画面和红/黄/绿/蓝 HSV 过滤画面。**

使用前请停止其它摄像头程序。本 Noimport subprocess

# 强制终止所有占用摄像头的进程
subprocess.run(["sudo", "-n", "fuser", "-k", "/dev/video0"])
subprocess.run(["sudo", "-n", "killall", "python3"])tebook 的两个实时模式单元**不要同时运行**。若摄像头报忙，先中断旧的采集单元，确认 `/dev/video0` 已释放。不要使用 `cv2.imshow()`。

In [5]:
import os
import signal
import subprocess
import time

CAMERA = "/dev/video1"
CURRENT_PID = os.getpid()

# 先释放当前 Notebook 之前打开的摄像头
if "cap" in globals():
    try:
        cap.release()
    except Exception:
        pass

# 获取占用摄像头的进程
result = subprocess.run(
    ["fuser", CAMERA],
    capture_output=True,
    text=True
)

pids = [
    int(pid)
    for pid in result.stdout.split()
    if pid.isdigit()
]

print("当前 Notebook PID:", CURRENT_PID)
print("摄像头占用进程:", pids)

for pid in pids:

    # 不终止当前 Notebook
    if pid == CURRENT_PID:
        print(f"保留当前 Notebook: {pid}")
        continue

    try:
        print(f"正在终止摄像头占用进程: {pid}")
        os.kill(pid, signal.SIGTERM)

    except ProcessLookupError:
        print(f"进程 {pid} 已退出")

    except PermissionError:
        print(f"权限不足，无法终止 {pid}")

time.sleep(1)

# 再次检查
subprocess.run(["fuser", "-v", CAMERA])

print("摄像头清理操作完成")

当前 Notebook PID: 20607
摄像头占用进程: []
摄像头清理操作完成


## 1. 导入依赖、设置四色 HSV 阈值
OpenCV 的 H 为 0–179，S/V 为 0–255。红色需要两段 H 范围；下列参数是初值，实际需要根据现场光照标定。

In [6]:
import cv2
import time
import json
from pathlib import Path
import numpy as np
import ipywidgets as widgets
from IPython.display import display

print('OpenCV:', cv2.__version__)

HSV_RANGES = {
    'red':    [((0, 95, 65), (10, 255, 255)), ((170, 95, 65), (179, 255, 255))],
    'yellow': [((18, 85, 65), (38, 255, 255))],
    'green':  [((39, 65, 50), (85, 255, 255))],
    'blue':   [((90, 75, 50), (135, 255, 255))],
}

CAMERA_INDEX = 0
FRAME_WIDTH, FRAME_HEIGHT = 640, 480  # 摄像头不一定支持，设置失败时沿用默认值
FPS_LIMIT = 8                         # 浏览器端显示帧率上限，不是摄像头物理帧率
JPEG_QUALITY = 70

OUTPUT_DIR = Path('week5_results')
OUTPUT_DIR.mkdir(exist_ok=True)


OpenCV: 4.5.1


## 2. 图像处理函数（不产生额外图像窗口）
根据 HSV 阈值生成四色 Mask，合并后在原始帧中保留对应像素。

In [7]:
def bgr8_to_jpeg(frame, quality=JPEG_QUALITY):
    ok, encoded = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, quality])
    if not ok:
        raise RuntimeError('JPEG 编码失败')
    return encoded.tobytes()


def filter_four_colors(frame_bgr, ranges=HSV_RANGES):
    hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
    combined_mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
    for intervals in ranges.values():
        for lower, upper in intervals:
            mask = cv2.inRange(hsv, np.array(lower, dtype=np.uint8),
                                  np.array(upper, dtype=np.uint8))
            combined_mask = cv2.bitwise_or(combined_mask, mask)
    filtered = cv2.bitwise_and(frame_bgr, frame_bgr, mask=combined_mask)
    return filtered, combined_mask


## 3. 双画面实时摄像头（运行本单元）

使用与参考文件一致的 `widgets.Image` 展示方式，不调用 `cv2.imshow()`。

- 默认演示 **30 秒**，可修改 `RUN_SECONDS`；设置为 `0` 表示持续运行，直到手动点击 Jupyter 的 **Interrupt Kernel**。
- 只在下面显示 **原始画面**和**四色过滤画面**；处理中的 HSV / Mask 不另外显示。
- 如果摄像头读取失败，连续重试有限次数后退出，不会把空帧传给编码器。
- **不要与其它摄像头采集单元并行运行。**

In [8]:
RUN_SECONDS = 30  # 改为 0 可持续运行，手动 Interrupt Kernel 退出

original_widget = widgets.Image(format='jpeg', width=480, height=360)
filtered_widget = widgets.Image(format='jpeg', width=480, height=360)
display(widgets.HBox([
    widgets.VBox([widgets.HTML('<b>原始摄像头画面</b>'), original_widget]),
    widgets.VBox([widgets.HTML('<b>红黄绿蓝过滤画面</b>'), filtered_widget]),
]))

cap = cv2.VideoCapture(CAMERA_INDEX)  # 沿用参考 Notebook 的调用方式
start_time = time.monotonic()
consecutive_failures = 0
last_original = None
last_filtered = None
try:
    if not cap.isOpened():
        raise RuntimeError('无法打开摄像头；先检查其他 Notebook 是否占用 /dev/video0')

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)
    print('摄像头已打开；实际分辨率:', int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
          'x', int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))

    while RUN_SECONDS == 0 or time.monotonic() - start_time < RUN_SECONDS:
        frame_start = time.monotonic()
        ret, frame = cap.read()
        if not ret or frame is None:
            consecutive_failures += 1
            if consecutive_failures >= 3:
                print('连续 3 次读取失败（可能是 select() timeout 或 USB 视频流异常），停止采集。')
                break
            time.sleep(0.2)
            continue
        consecutive_failures = 0

        filtered, mask = filter_four_colors(frame)
        original_widget.value = bgr8_to_jpeg(frame)
        filtered_widget.value = bgr8_to_jpeg(filtered)
        last_original, last_filtered = frame.copy(), filtered.copy()

        elapsed = time.monotonic() - frame_start
        time.sleep(max(0, 1.0/FPS_LIMIT - elapsed))
except KeyboardInterrupt:
    print('用户已停止实时采集')
finally:
    cap.release()
    print('摄像头已释放')


摄像头已释放


[ WARN:0] global /tmp/pip-req-build-q3gzfcr4/opencv/modules/videoio/src/cap_v4l.cpp (893) open VIDEOIO(V4L2:/dev/video0): can't open camera by index


RuntimeError: 无法打开摄像头；先检查其他 Notebook 是否占用 /dev/video0

## 4. 保存实验截图及当前 HSV 阈值
先完成第3节实时采集，再运行此单元。只保存文件，不增加显示画面。

In [ ]:
if last_original is None or last_filtered is None:
    print('暂无成功采集的画面，请先运行第3节。')
else:
    stamp = time.strftime('%Y%m%d_%H%M%S')
    cv2.imwrite(str(OUTPUT_DIR / f'{stamp}_original.jpg'), last_original)
    cv2.imwrite(str(OUTPUT_DIR / f'{stamp}_filtered.jpg'), last_filtered)
    with (OUTPUT_DIR / f'{stamp}_hsv.json').open('w', encoding='utf-8') as f:
        json.dump(HSV_RANGES, f, ensure_ascii=False, indent=2)
    print('已保存原始图、过滤图、HSV 参数到:', OUTPUT_DIR.resolve())


## 5. 不同光照下的 HSV 标定与实验记录

在第1节调整 `HSV_RANGES` 中的阈值，再运行第3节观察过滤效果。分别在正常、暗光、强光环境下测试，每次运行第4节保存图像及阈值。下面单元可把人工评估记录追加到 CSV。识别效果必须填写真实观察值，不预填实验结果。

In [ ]:
import csv

LIGHTING = 'normal'  # normal / dim / bright
TEST_COLOR = 'red'   # red / yellow / green / blue
RESULT = ''         # 填写：正确 / 漏检 / 误检 / 部分检出
NOTES = ''          # 例如：阴影导致饱和度降低

if not RESULT:
    print('请先填写 RESULT，再执行记录。')
else:
    record_path = OUTPUT_DIR / 'lighting_tests.csv'
    row = {
        'time': time.strftime('%Y-%m-%d %H:%M:%S'),
        'lighting': LIGHTING,
        'target_color': TEST_COLOR,
        'result': RESULT,
        'notes': NOTES,
        'hsv_ranges': json.dumps(HSV_RANGES, ensure_ascii=False),
    }
    new_file = not record_path.exists()
    with record_path.open('a', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(row))
        if new_file:
            writer.writeheader()
        writer.writerow(row)
    print('记录已追加到:', record_path)


## 使用注意事项

1. `VideoCapture(0)` 是参考文件的做法；如 USB 摄像头编号变化，应根据 `v4l2-ctl --list-devices` 检查。不要在两个 Notebook 中同时运行采集循环。
2. 实时循环用 `try...finally` 包裹 `cap.release()`。**如果强制关闭进程或内核崩溃，仍需检查设备是否释放。**
3. `ipywidgets.Image(format="jpeg")` 适合 JupyterLab 浏览器显示；Jetson Nano 性能有限，刷新率不必过高。
4. 如果 `cap.read()` 出现 `select() timeout`，通常不是显示控件问题，需检查摄像头格式、USB 连接和视频流状态。
5. HSV 阈值必须在现场标定；红色是两段 Hue。当前没有加方块形状筛选，画面中任何符合 HSV 阈值的红黄绿蓝区域都可能保留。